# ex03 · 线性回归的简洁实现（对应教材 3.3）

> **做题流程**：补全 TODO，每步运行自测；做完再看 `solutions/ex03-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节用 nn 模块把 ex02 的三样东西（模型/损失/优化器）各换成一个对象，并对比框架替你省了什么。

In [2]:
import random
import torch
from torch import nn
from torch.utils import data

def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

def load_array(data_arrays, batch_size, is_train=True):
    """把张量打包成 DataLoader"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)
batch_size = 10
data_iter = load_array((features, labels), batch_size)

## 题 1 🔧 填空：nn 三件套

nn 模块把 ex02 里手写的三样东西各换成一个对象。补全优化器，并先思考：nn.Linear / nn.MSELoss / SGD 优化器分别替换了 ex02 里的哪些代码？

**【你的预测】**

In [5]:
net = nn.Sequential(nn.Linear(2, 1))
loss = nn.MSELoss()

# TODO 3.1: 创建 SGD 优化器，学习率 0.03，优化 net 的全部参数
trainer = torch.optim.SGD(net.parameters(), lr = 0.03)

### 读代码：nn.Linear 里有什么（运行查看）

In [6]:
print('net 结构:', net)
print('weight 形状:', list(net[0].weight.shape), ' bias 形状:', list(net[0].bias.shape))

net 结构: Sequential(
  (0): Linear(in_features=2, out_features=1, bias=True)
)
weight 形状: [1, 2]  bias 形状: [1]


## 题 2 🔧 补全训练循环（TODO 3.2 ~ 3.4）

对比 ex02 的训练循环：梯度清零、反向传播、参数更新，这里各由哪个对象负责？

**【你的预测】**

In [7]:
num_epochs = 3
try:
    for epoch in range(num_epochs):
        for X, y in data_iter:
            l = loss(net(X), y)
            # TODO 3.2: 清零梯度（用哪个对象？）
            trainer.zero_grad()
            # TODO 3.3: 反向传播
            l.backward()
            # TODO 3.4: 更新参数
            trainer.step()
        l = loss(net(features), labels)
        print(f'epoch {epoch + 1}, loss {l:f}')
except NotImplementedError as e:
    print(f'⚠ {e}，先完成 TODO 3.2~3.4 再运行')
except NameError as e:
    print(f'⚠ 名字未定义: {e}，先完成 TODO 3.1')

epoch 1, loss 0.000193
epoch 2, loss 0.000090
epoch 3, loss 0.000090


## 题 3 🔧 变式：换优化器 Adam

把 SGD 换成 Adam（lr 用 0.1），其余不动。先预测：Adam 前 3 个 epoch 的 loss 会比 SGD 低吗？为什么？

**【你的预测】**

In [9]:
try:
    net2 = nn.Sequential(nn.Linear(2, 1))
    loss2 = nn.MSELoss()
    # TODO 3.5: 创建 Adam 优化器（lr=0.1），替换下面这一行
    trainer2 = torch.optim.Adam(net2.parameters(), lr = 0.1)
    num_epochs = 3
    for epoch in range(num_epochs):
        for X, y in data_iter:
            l2 = loss2(net2(X), y)
            trainer2.zero_grad()
            l2.backward()
            trainer2.step()
        print(f'Adam epoch {epoch + 1}, loss {loss2(net2(features), labels):f}')
except NotImplementedError as e:
    print(f'⚠ {e}')

Adam epoch 1, loss 0.000371
Adam epoch 2, loss 0.000091
Adam epoch 3, loss 0.000090


### 对比两种优化器训练后的参数

In [10]:
print('SGD  网络参数:', net[0].weight.detach().numpy().round(4), ' bias:', round(net[0].bias.item(), 4))
print('Adam 网络参数:', net2[0].weight.detach().numpy().round(4), ' bias:', round(net2[0].bias.item(), 4))
print('真实参数: [[ 2.  -3.4]]  4.2')

SGD  网络参数: [[ 1.999  -3.3997]]  bias: 4.2004
Adam 网络参数: [[ 2.0007 -3.3996]]  bias: 4.1998
真实参数: [[ 2.  -3.4]]  4.2


## 题 4 🌱 问答：框架替你做了什么？

先写你的回答，再对照答案文件：

1. nn.Linear(2, 1) 内部有什么参数？它替 ex02 做了哪件事？

w，b， inputchannel， outputchannel
weight (1,2) + bias (1)，自动随机的初始化；它替 ex02 做了「创建并初始化参数」这件事。
2. 训练循环里的梯度清零、反向、更新，分别对应 ex02 的哪些代码行？

param.zero_grad()
loss.sum.backward()
param -= lr * param.grad / batch_size
3. 既然框架这么方便，为什么还要学 ex02 的从零实现？（面试高频）

弄清原理，确定底层逻辑
从零实现暴露了「参数在哪、梯度怎么来、怎么更新」的全部细节。后续自定义损失、自定义层、调试训练问题时，都要靠这些细节
**【你的预测】**

## 小结与面试衔接

- 简洁实现 = 模型（nn.Linear）+ 损失（nn.MSELoss）+ 优化器（optim.SGD）三个对象
- 训练三步：trainer.zero_grad() → l.backward() → trainer.step()，与从零版一一对应
- Adam 自适应学习率，通常比固定 lr 的 SGD 收敛快（ch11 展开，面试高频）
- 面试高频：SGD 与 Adam 的区别——Adam 用梯度的一阶/二阶矩估计给每个参数自适应步长